# 🤖 CloudSync SaaS Analytics: 04. Churn Risk Modeling & Decision Intelligence

**Project:** CloudSync Customer Churn & Retention Analytics  
**Dataset:** CloudSync Master Customer Profile  
**Objective:** Develop interpretable machine learning models (Logistic Regression baseline vs. Random Forest Classifier) to estimate customer-level churn probabilities, evaluate feature importance / odds ratios, stratify risk categories, and establish targeted retention triggers.

---

### Methodological Guardrails:
- **No Data Leakage**: We isolate features that precede churn events and apply `ColumnTransformer` / `Pipeline` structures strictly fitted on the training split.
- **Interpretable Risk Stratification**:
  - 🟢 **Low Risk**: $\text{Predicted Churn Probability} < 0.35$
  - 🟡 **Medium Risk**: $0.35 \le \text{Predicted Churn Probability} < 0.70$
  - 🔴 **High Risk**: $\text{Predicted Churn Probability} \ge 0.70$

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 300

CLEAN_DIR = '../data/cleaned'
df = pd.read_csv(os.path.join(CLEAN_DIR, "customer_churn_master.csv"))

print(f"Master dataset loaded with {len(df)} records.")

## 1. Feature Definition & Train/Test Partition

In [ ]:
df['target_churn'] = (df['subscription_status'] == 'Churned').astype(int)

feature_cols_num = [
    'tenure_months', 'monthly_fee', 'discount_percentage',
    'avg_monthly_logins', 'avg_active_days', 'avg_session_mins',
    'avg_features_used', 'ticket_count', 'unresolved_tickets',
    'failed_payments', 'nps_score', 'feedback_csat'
]
feature_cols_cat = ['contract_type', 'plan_name', 'customer_segment', 'acquisition_channel', 'auto_renew']

X = df[feature_cols_num + feature_cols_cat]
y = df['target_churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set:  {X_test.shape[0]} samples")
print(f"Baseline Churn Rate in Train: {y_train.mean()*100:.2f}%")

## 2. Model Training: Logistic Regression Baseline & Random Forest

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), feature_cols_num),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), feature_cols_cat)
    ]
)

# 1. Logistic Regression
lr_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42, C=1.0))
])
lr_pipe.fit(X_train, y_train)

# 2. Random Forest
rf_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=150, max_depth=10, min_samples_leaf=5, random_state=42))
])
rf_pipe.fit(X_train, y_train)

print("Both models trained successfully.")

## 3. Comprehensive Performance Benchmark & ROC-AUC

In [ ]:
y_pred_lr = lr_pipe.predict(X_test)
y_prob_lr = lr_pipe.predict_proba(X_test)[:, 1]

y_pred_rf = rf_pipe.predict(X_test)
y_prob_rf = rf_pipe.predict_proba(X_test)[:, 1]

results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest Classifier'],
    'Accuracy': [accuracy_score(y_test, y_pred_lr), accuracy_score(y_test, y_pred_rf)],
    'Precision': [precision_score(y_test, y_pred_lr), precision_score(y_test, y_pred_rf)],
    'Recall': [recall_score(y_test, y_pred_lr), recall_score(y_test, y_pred_rf)],
    'F1 Score': [f1_score(y_test, y_pred_lr), f1_score(y_test, y_pred_rf)],
    'ROC-AUC': [roc_auc_score(y_test, y_prob_lr), roc_auc_score(y_test, y_prob_rf)]
})

display(results.round(4))

## 4. Confusion Matrix & ROC Curves Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curves
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)

axes[0].plot(fpr_lr, tpr_lr, label=f"Logistic Regression (AUC = {roc_auc_score(y_test, y_prob_lr):.3f})", color='#e67e22', lw=2)
axes[0].plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC = {roc_auc_score(y_test, y_prob_rf):.3f})", color='#2980b9', lw=2)
axes[0].plot([0, 1], [0, 1], 'k--', lw=1.2)
axes[0].set_title("ROC Curves Comparison", fontweight='bold', fontsize=12)
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend(loc='lower right')

# Confusion Matrix RF
cm = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[1],
            xticklabels=['Retained', 'Churned'], yticklabels=['Retained', 'Churned'])
axes[1].set_title("Confusion Matrix (Random Forest)", fontweight='bold', fontsize=12)
axes[1].set_xlabel("Predicted Label")
axes[1].set_ylabel("Actual Label")

plt.tight_layout()
plt.show()

## 5. Interpretability: Feature Importance & Odds Ratios

In [ ]:
cat_encoder = lr_pipe.named_steps['preprocessor'].named_transformers_['cat']
cat_feature_names = list(cat_encoder.get_feature_names_out(feature_cols_cat))
all_features = feature_cols_num + cat_feature_names

# Logistic Regression Odds Ratios
df_coefs = pd.DataFrame({
    'Feature': all_features,
    'Log_Odds_Coef': lr_pipe.named_steps['classifier'].coef_[0],
    'Odds_Ratio': np.exp(lr_pipe.named_steps['classifier'].coef_[0])
}).sort_values(by='Log_Odds_Coef', ascending=False)

print("Top 5 Factors Increasing Churn Odds:")
display(df_coefs.head(5))

print("
Top 5 Protective Factors Reducing Churn Odds:")
display(df_coefs.tail(5))

## 6. Actionable Risk Table: High-Risk Customers for Retention Intervention

In [ ]:
# Identify Active High Risk Accounts
at_risk_active = df[(df['subscription_status'] == 'Active') & (df['risk_category'] == 'High Risk')].sort_values(by='monthly_fee', ascending=False)

print(f"Total Active High-Risk Customers: {len(at_risk_active):,}")
print(f"Total Active MRR at Immediate Risk: ${at_risk_active['monthly_fee'].sum():,.2f}")

display(at_risk_active[['customer_id', 'customer_name', 'plan_name', 'contract_type', 'monthly_fee', 'tenure_months', 'avg_monthly_logins', 'failed_payments', 'unresolved_tickets', 'churn_probability', 'risk_category']].head(15))